# P5. 보도자료 요약 · 검색기

보건복지부 2026년 10월 보도자료 PDF 83건을 MinIO 버킷에 모아 텍스트를 뽑고, 청킹 · 임베딩 · Qdrant 로 「이번 달 ○○ 정책 요약해줘」에 근거 문단과 함께 답합니다.

- 단계: 개별 프로젝트 · 비정형 · 예상 2~3일
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 보건복지부 — 보도자료 첨부 PDF 2026-10-01~10-28 게시분 83건(2026-10-28 수집, 게시물마다 공공누리 제1유형(출처표시) 확인)

**빈칸 노트북** — 코드 속 `___①___` 자리 16곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P5/P5_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

보도자료는 정부 정책이 처음 발표되는 자리입니다. 하지만 PDF 로 흩어져 있어 「이번 달 노인 돌봄 정책이 뭐가 나왔지?」를 알려면 수십 개 파일을 열어 봐야 합니다.
이 프로젝트에서는 한 달치 보도자료 PDF 를 모아 **물어보면 관련 보도자료를 찾아 요약해 주는 검색기**를 만듭니다.

4주차의 레이크(MinIO)와 7주차의 RAG 를 한 줄로 잇습니다 — 7주차 7-4 「MinIO·Qdrant 연동」을 보도자료로 끝까지 해 보는 셈입니다.

```text
보도자료 PDF(수집) → MinIO 버킷 press-raw → 텍스트 추출(pypdf) → 버킷 press-text
  → 청킹 → 임베딩(ko-sroberta) → Qdrant → 검색(이번 달 필터) → 요약(gpt-4o-mini, 키가 있을 때)
```

다 끝내면 다음을 말할 수 있습니다.

- 원본 PDF 와 뽑아낸 텍스트를 왜 **버킷을 나눠** 두는지
- 검색 결과에 「이번 달」 같은 조건을 거는 방법(payload 필터)
- LLM 이 없어도 검색기가 할 수 있는 일과, LLM 을 붙이면 달라지는 일

## 1단계 · 준비 — 라이브러리와 데이터

Colab 에 없는 라이브러리를 먼저 설치합니다. 로컬에 이미 있으면 건너뜁니다.

데이터는 보건복지부 누리집 「보도자료」 게시판에 2026년 10월 1일~28일 올라온 83건의 첨부 PDF 입니다(게시물마다 첫 번째 PDF).
83건 모두 게시물 아래에 **공공누리 제1유형(출처표시)** 이 붙어 있어 출처를 밝히면 다시 나눠 쓸 수 있습니다. 파일이 커서 날짜로 두 묶음(1~15일 · 16~28일)으로 나눴습니다.
목록 파일 `mohw_press_202610.jsonl` 에는 게시물마다 제목 · 날짜 · 담당 부서 · 원문 주소가 들어 있습니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("boto3")
need("pypdf")
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")
need("langchain_text_splitters", "langchain-text-splitters")

import os, io, re, json, time, shutil, zipfile, platform, urllib.request
import datetime as dt
import requests
import pandas as pd

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

meta = [json.loads(l) for l in open(fetch("mohw_press_202610.jsonl"), encoding="utf-8")]
for z in ["mohw_press_202610_a.zip", "mohw_press_202610_b.zip"]:
    zipfile.ZipFile(fetch(z)).extractall("data/pdf")
pdfs = sorted(os.listdir("data/pdf"))
print("목록:", len(meta), "건 | PDF:", len(pdfs), "개 |", round(sum(os.path.getsize("data/pdf/" + p) for p in pdfs) / 1e6, 1), "MB")
pd.DataFrame(meta)[["doc_id", "date", "title", "dept"]].head(3)

> **확인 포인트** — `목록: 83 건 | PDF: 83 개 | 37.4 MB` 가 찍힙니다.

In [ ]:
assert len(meta) == 83 and len(pdfs) == 83
assert {m["file"] for m in meta} == set(pdfs)

## 2단계 · 직접 모으기 — 게시판에서 PDF 받아 보기(실시간)

스냅샷은 이 단계와 같은 방법으로 모았습니다. 게시판 목록 → 게시물 화면 → 첨부 PDF 순서로 따라갑니다(2주차 11장).
먼저 robots.txt 로 목록 · 첨부 주소를 가져가도 되는지 확인합니다.

In [ ]:
from urllib.robotparser import RobotFileParser

UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/130 Safari/537.36"}
BASE = "https://www.mohw.go.kr"
LIST = BASE + "/board.es?mid=a10503000000&bid=0027&nPage=1"

online = True
try:
    rp = RobotFileParser()
    rp.parse(requests.get(BASE + "/robots.txt", headers=UA, timeout=20).text.splitlines())
    print("목록:", rp.can_fetch("*", LIST), "| 첨부:", rp.can_fetch("*", BASE + "/boardDownload.es?bid=0027"))
except requests.RequestException as e:
    online = False
    print("접속되지 않습니다:", type(e).__name__, "— 스냅샷으로 계속하세요.")

목록 화면에서 게시물 번호(`list_no`)를 뽑고, 최근 세 건만 들어가 첫 번째 PDF 를 받습니다. 요청 사이에는 쉬어 줍니다.

In [ ]:
live = {}
if online:
    html = requests.get(LIST, headers=UA, timeout=30).text
    nos = re.___①___(r"list_no=(\d+)&amp;tag=&amp;nPage=1\" class=\"txt_title\"", html)   # ① 규칙에 맞는 조각을 모두 찾기
    print("목록 1쪽 게시물:", len(nos), "건", nos[:3])
    for no in nos[:3]:
        view = requests.get(f"{BASE}/board.es?mid=a10503000000&bid=0027&list_no={no}&act=view", headers=UA, timeout=30).text
        m = re.search(r'href="(/boardDownload\.es\?[^"]+)" title="[^"]*\.pdf"', view)
        if m:
            body = requests.get(BASE + m.group(1).replace("&amp;", "&"), headers=UA, timeout=60).content
            live[f"mohw_{no}"] = body
            print(no, round(len(body) / 1e3), "KB", body[:5])
        time.sleep(1)

> **확인 포인트** — 목록 1쪽에서 게시물 번호가 15건쯤 나오고(2026-10-28 실행 15건), 받은 파일마다 앞 다섯 글자가 `b'%PDF-'` 입니다. PDF 파일은 늘 이 다섯 글자로 시작합니다.
> 실시간이라 번호와 크기는 날마다 다릅니다.

In [ ]:
assert all(b[:5] == b"%PDF-" for b in live.values())

> **자주 겪는 오류** — Colab 서버는 해외에 있어 기관 누리집이 접속을 막을 수 있습니다(`ConnectTimeout`). 그러면 이 단계는 읽기만 하고 스냅샷으로 계속하세요.
> 수집 코드를 고칠 일이 생기면 브라우저에서 게시물 화면을 열고 「페이지 소스 보기」로 첨부 링크 모양을 먼저 확인하세요. 누리집 개편 때 가장 먼저 바뀌는 곳입니다.

## 3단계 · MinIO 버킷에 올리기

원본 PDF 는 고치지 않고 그대로 보관합니다. 오브젝트 스토리지(MinIO)는 이런 원본 파일을 담는 곳입니다(4주차 17장).
Colab 에는 Docker 가 없으니 MinIO 서버 프로그램을 받아 백그라운드로 띄웁니다. 과정에서 쓴 것과 같은 판(pgsty/minio `RELEASE.2026-08-04T00-00-00Z`)입니다.

In [ ]:
import socket, tarfile

PORT = 9100                                       # 9000 은 내 컴퓨터의 Docker MinIO 가 쓰고 있을 수 있어 비켜 둡니다
osname = platform.system().lower()                # linux(Colab) · darwin(macOS)
arch = "arm64" if platform.machine().lower() in ("arm64", "aarch64") else "amd64"
TGZ = ("https://github.com/pgsty/silo/releases/download/RELEASE.2026-08-04T00-00-00Z/"
       f"minio_20260804000000.0.0_{osname}_{arch}.tar.gz")

def port_open(p):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", p)) == 0

minio_proc = None
try:
    if port_open(PORT):
        raise RuntimeError(f"{PORT} 포트를 이미 누가 쓰고 있습니다")
    if not os.path.exists("bin/minio"):
        os.makedirs("bin", exist_ok=True)
        urllib.request.urlretrieve(TGZ, "bin/minio.tgz")
        with tarfile.open("bin/minio.tgz") as t:
            t.extract("minio", "bin")
        os.chmod("bin/minio", 0o755)
    env = dict(os.environ, MINIO_ROOT_USER="minioadmin", MINIO_ROOT_PASSWORD="minioadmin")
    minio_proc = subprocess.___②___(   # ② 끝나기를 기다리지 않고 띄우기
        ["bin/minio", "server", "minio-data", "--address", f"127.0.0.1:{PORT}", "--console-address", f"127.0.0.1:{PORT + 1}"],
        env=env, stdout=open("minio.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):                           # 뜰 때까지 최대 30초 기다립니다
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/minio/health/live", timeout=1).status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        raise RuntimeError("MinIO 가 30초 안에 뜨지 않았습니다 — minio.log 를 보세요")
    print("MinIO 실행 중:", f"http://127.0.0.1:{PORT}")
except Exception as e:
    minio_proc = None
    print("MinIO 를 띄우지 못했습니다:", e, "→ 로컬 폴더를 버킷으로 씁니다")

S3 명령은 boto3 로 보냅니다(4주차 17장). MinIO 를 띄우지 못했으면 같은 이름의 함수가 `lake/` 폴더를 버킷처럼 쓰게 해 둡니다 — 뒤 단계 코드는 어느 쪽이든 같습니다.

In [ ]:
import boto3

if minio_proc:
    s3 = boto3.client("s3", endpoint_url=f"http://127.0.0.1:{PORT}",
                      aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin", region_name="us-east-1")
    def make_bucket(b):
        if b not in [x["Name"] for x in s3.list_buckets()["Buckets"]]:
            s3.create_bucket(Bucket=b)
    def put(b, key, body, meta=None):
        s3.___③___(Bucket=b, Key=key, Body=body, Metadata=meta or {})   # ③ 객체 하나 올리기
    def get(b, key):
        return s3.get_object(Bucket=b, Key=key)["Body"].read()
    def ls(b, prefix=""):
        out = []
        for page in s3.get_paginator("___④___").paginate(Bucket=b, Prefix=prefix):   # ④ 객체 목록 API
            out += [(o["Key"], o["Size"]) for o in page.get("Contents", [])]
        return out
else:                                             # 대안: lake/<버킷>/<키> 폴더가 곧 버킷
    def make_bucket(b):
        os.makedirs(f"lake/{b}", exist_ok=True)
    def put(b, key, body, meta=None):
        if not os.path.isdir(f"lake/{b}"):
            raise FileNotFoundError(f"NoSuchBucket: {b}")
        os.makedirs(os.path.dirname(f"lake/{b}/{key}"), exist_ok=True)
        open(f"lake/{b}/{key}", "wb").write(body)
    def get(b, key):
        return open(f"lake/{b}/{key}", "rb").read()
    def ls(b, prefix=""):
        root = f"lake/{b}/"
        files = [os.path.join(d, f)[len(root):] for d, _, fs in os.walk(root) for f in fs]
        return sorted((k, os.path.getsize(root + k)) for k in files if k.startswith(prefix))
print("버킷 방식:", "MinIO(S3)" if minio_proc else "로컬 폴더")

> **Aebon's Tip** — MinIO 가 뜨지 않아 로컬 폴더로 진행했어도 키 구조(`<버킷>/mohw/2026-10/…`)가 같아서 나중에 그대로 옮길 수 있습니다.
> 내 컴퓨터의 MinIO 로 옮길 때는 4주차의 mc 로 `mc mirror lake/press-raw local/press-raw` 처럼 폴더를 버킷에 맞춰 올리세요.

일부러 실패해 봅니다. 버킷을 만들기 전에 올리면 어떻게 될까요?

In [ ]:
try:
    put("press-raw", "test.txt", b"hello")
except Exception as e:
    print("실패:", type(e).__name__, str(e)[:80])

`NoSuchBucket` 입니다. S3 는 폴더처럼 저절로 생기지 않습니다 — 버킷은 먼저 만들어야 합니다.
이제 버킷을 만들고 83개 PDF 를 올립니다. 키는 `mohw/2026-10/<doc_id>.pdf` 로 기관 · 달을 앞에 둡니다. 나중에 「이 기관 이 달」만 골라 읽기 쉽습니다.

In [ ]:
___⑤___("press-raw")   # ⑤ 버킷 만들기
for m in meta:
    body = open("data/pdf/" + m["file"], "rb").read()
    put("press-raw", f"mohw/{m['date'][:7]}/{m['doc_id']}.pdf", body, {"date": m["date"], "doc_id": m["doc_id"]})

raw_objs = ls("press-raw", prefix="mohw/2026-10/")
print("press-raw:", len(raw_objs), "개 |", round(sum(s for _, s in raw_objs) / 1e6, 1), "MB")
raw_objs[:3]

> **확인 포인트** — `press-raw: 83 개 | 37.4 MB`, 키는 `mohw/2026-10/mohw_1492111.pdf` 처럼 들어갑니다.

In [ ]:
assert len(raw_objs) == 83
assert raw_objs[0][0] == "mohw/2026-10/mohw_1492111.pdf"

> **Aebon's Tip** — 메타데이터(`Metadata=`)에는 한글을 넣을 수 없습니다. HTTP 헤더로 가기 때문에 S3 규격이 ASCII 만 받고, boto3 가 `ParamValidationError` 로 먼저 막습니다.
> 제목처럼 한글인 정보는 목록 파일(jsonl)에 두고 `doc_id` 로 잇습니다.

> **Aebon's Tip** — 내 컴퓨터에서는 4주차의 MinIO 컨테이너(`localhost:9000`, 콘솔 9001)에 그대로 올려도 됩니다. `endpoint_url` 과 키만 바꾸면 이 노트북의 나머지가 그대로 돕니다.
> 콘솔에서 `press-raw` 버킷을 열어 PDF 가 들어간 것을 눈으로 확인해 보세요.

## 4단계 · 텍스트 추출 — 원본 버킷에서 읽어 텍스트 버킷으로

PDF 에서 글자를 뽑습니다. `pypdf` 의 `extract_text()` 는 쪽마다 글자를 돌려줍니다.
먼저 일부러 실패해 봅니다. 2단계에서 첨부 링크 대신 게시물 화면(HTML)을 받았다면 이런 일이 생깁니다.

In [ ]:
from pypdf import PdfReader
from pypdf.errors import PdfReadError

try:
    PdfReader(io.BytesIO("<!DOCTYPE html><html><body>보도자료</body></html>".encode("utf-8")))
except PdfReadError as e:
    print("PDF 가 아닙니다:", e)

> **확인 포인트** — `PdfReadError` 가 납니다. 받은 파일이 정말 PDF 인지는 앞 다섯 글자(`%PDF-`)로 먼저 확인하는 것이 좋습니다.

쪽 번호 줄(`- 1 -`)을 지우고, 줄바꿈과 여러 칸 공백을 한 칸으로 줄입니다. 83개를 읽는 데 1~2분 걸립니다.

In [ ]:
def pdf_text(body):
    reader = PdfReader(io.BytesIO(body))
    raw = "\n".join(p.___⑥___() or "" for p in reader.pages)   # ⑥ 쪽에서 글자 뽑기
    raw = re.sub(r"^\s*-\s*\d+\s*-\s*$", "", raw, flags=re.M)     # 쪽 번호 줄
    return len(reader.pages), re.sub(r"\s+", " ", raw).strip()

t0 = time.time()
texts = {}
for key, _ in raw_objs:
    doc_id = key.split("/")[-1][:-4]
    pages, text = pdf_text(___⑦___)   # ⑦ 버킷에서 원본 읽기
    texts[doc_id] = text
print(len(texts), "건 |", sum(len(t) for t in texts.values()), "자 |", round(time.time() - t0), "초")
print(texts["mohw_1492342"][:300])

> **확인 포인트** — 83건, 모두 **361,824자**입니다. 가장 긴 보도자료는 `mohw_1492210`(22,758자), 10/28 건강보험 수가 보도자료 `mohw_1492342` 는 20,435자입니다.

In [ ]:
assert len(texts) == 83 and sum(len(t) for t in texts.values()) == 361824
assert len(texts["mohw_1492342"]) == 20435
assert max(texts, key=lambda k: len(texts[k])) == "mohw_1492210"

> **자주 겪는 오류** — 부제목 일부가 `지 역·필 수 의 료` 처럼 한 글자씩 띄어져 나옵니다. PDF 가 글자 간격을 넓혀 그린 줄은 pypdf 가 글자마다 띄어 읽기 때문입니다.
> 임베딩 검색은 이 정도는 견디지만, 키워드 검색(7주차 하이브리드)을 붙일 때는 정제가 필요합니다.

뽑은 텍스트는 원본과 다른 버킷 `press-text` 에 JSONL 로 둡니다. 추출 규칙을 바꾸면 원본에서 다시 뽑으면 됩니다 — 원본을 지키는 이유입니다(4주차 4-3 「원천 → 가공」 층 나누기).

In [ ]:
title = {m["doc_id"]: m for m in meta}
lines = [json.dumps({"doc_id": k, "date": title[k]["date"], "title": title[k]["title"], "text": v}, ensure_ascii=False)
         for k, v in texts.items()]
make_bucket("press-text")
put("press-text", "mohw/2026-10/texts.jsonl", "\n".join(lines).encode("utf-8"))
print(ls("press-text"))

## 5단계 · 청킹 — 검색할 조각으로 자르기

보도자료 한 건은 2만 자가 넘기도 합니다. 통째로 임베딩하면 여러 주제가 한 벡터에 섞여 검색이 흐려집니다. 500자 조각으로 자르고, 조각 사이를 50자 겹쳐 문장이 끊긴 자리도 이어지게 합니다(7주차 7-2).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=___⑧___, chunk_overlap=___⑨___,   # ⑧ 조각 최대 글자 수 · ⑨ 겹치는 글자 수
                                          separators=["\n\n", "\n", "다. ", ". ", " ", ""], keep_separator="end")
chunks = []
for m in meta:
    for i, c in enumerate(splitter.split_text(texts[m["doc_id"]])):
        chunks.append({"doc_id": m["doc_id"], "title": m["title"], "date": int(m["date"].replace("-", "")),
                       "i": i, "text": c})
print("조각:", len(chunks), "| 가장 긴 조각:", max(len(c["text"]) for c in chunks), "자")
chunks[4]

`keep_separator="end"` 는 자른 자리의 `다. ` 를 앞 조각 끝에 붙여 둡니다. 기본값이면 다음 조각이 `. ` 으로 시작합니다.
`date` 를 `20261028` 같은 숫자로 넣었습니다. 숫자는 「이상 · 이하」 범위 조건을 걸기 쉽습니다(7단계).

> **확인 포인트** — 조각 **998개**, 가장 긴 조각은 499자로 500자를 넘지 않습니다.

In [ ]:
assert len(chunks) == 998 and max(len(c["text"]) for c in chunks) <= 500

일부러 실패해 봅니다. 겹침이 조각보다 크면?

In [ ]:
try:
    RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=200)
except ValueError as e:
    print("설정 오류:", e)

> **Aebon's Tip** — `chunk_size` 를 300 · 1000 으로 바꿔 7단계 검색 결과를 비교해 보세요. 작으면 정확한 문장을 찾지만 앞뒤 맥락이 빠지고, 크면 맥락은 있지만 엉뚱한 내용이 섞입니다.

## 6단계 · 임베딩과 Qdrant 적재

조각마다 768차원 벡터를 만듭니다. 키 없이 도는 한국어 문장 임베딩 모델 `jhgan/ko-sroberta-multitask` 를 씁니다(7주차 7-3).
처음 실행할 때 모델(약 440MB)을 받느라 1~2분, 998개를 벡터로 바꾸는 데 1분 안팎 걸립니다.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jhgan/ko-sroberta-multitask")
t0 = time.time()
vecs = model.___⑩___([c["text"] for c in chunks], batch_size=32,   # ⑩ 글자를 벡터로
                    normalize_embeddings=True, show_progress_bar=False)
print(vecs.shape, round(time.time() - t0), "초")

> **확인 포인트** — `(998, 768)` — 조각 998개, 하나에 숫자 768개입니다.

In [ ]:
assert vecs.shape == (998, 768)

Qdrant 는 Colab 에서 메모리 안에 띄웁니다(`":memory:"`). 조각의 글 · 제목 · 날짜는 payload 로 함께 넣어, 검색 결과에서 바로 꺼내 쓰고 조건도 겁니다.

In [ ]:
from qdrant_client import QdrantClient, models

qd = QdrantClient(":memory:")
qd.create_collection("press", vectors_config=models.VectorParams(size=___⑪___, distance=models.Distance.___⑫___))   # ⑪ 벡터 길이 · ⑫ 방향이 얼마나 같은지
qd.upsert("press", points=[models.PointStruct(id=k, vector=vecs[k].tolist(), payload=c) for k, c in enumerate(chunks)])
print("적재:", qd.count("press").count)

> **확인 포인트** — 적재 **998**.

In [ ]:
assert qd.count("press").count == 998

일부러 실패해 봅니다. 다른 모델(예: OpenAI `text-embedding-3-small`, 1536차원)로 만든 벡터를 같은 컬렉션에 넣으면?

In [ ]:
try:
    qd.upsert("press", points=[models.PointStruct(id=99999, vector=[0.0] * 1536, payload={})])
except Exception as e:
    print("넣지 못했습니다:", type(e).__name__, str(e)[:90])
print("그대로:", qd.count("press").count)

> **자주 겪는 오류** — 컬렉션의 벡터 길이는 만들 때 정해집니다. 임베딩 모델을 바꾸면 컬렉션을 새로 만들고 전부 다시 넣어야 합니다 — 검색할 때의 질문도 **같은 모델**로 바꿔야 합니다.

> **Aebon's Tip** — 내 컴퓨터에서는 7주차의 Qdrant 컨테이너(`QdrantClient(url="http://localhost:6333")`)에 넣으면 노트북을 껐다 켜도 남아 있고, 대시보드(`:6333/dashboard`)에서 점을 볼 수 있습니다.

## 7단계 · 검색 — 「이번 달」 조건 걸기

질문도 같은 모델로 벡터로 바꿔 가까운 조각을 찾습니다. 「이번 달」은 날짜 payload 에 범위 조건(`Range`)을 걸어 풉니다.
기준일은 스냅샷을 모은 2026년 10월 28일로 고정합니다. 직접 모은 자료로 할 때는 `dt.date.today()` 로 바꾸세요.

In [ ]:
TODAY = dt.date(2026, 10, 28)

def month_range(day):
    first = day.replace(day=1)
    nxt = (first + dt.timedelta(days=32)).replace(day=1)
    return int(first.strftime("%Y%m%d")), int((nxt - dt.timedelta(days=1)).strftime("%Y%m%d"))

def search(q, k=8, month=TODAY):
    lo, hi = month_range(month)
    flt = models.Filter(must=[models.FieldCondition(key="date", range=models.___⑬___(gte=lo, lte=hi))])   # ⑬ 이상 · 이하 조건
    qv = model.encode(q, normalize_embeddings=True)
    return qd.query_points("press", query=qv.tolist(), limit=k, query_filter=___⑭___).points   # ⑭ 날짜 조건

print(month_range(TODAY))
for h in search("의료 수가 정책", k=5):
    print(round(h.score, 3), h.payload["doc_id"], h.payload["date"], h.payload["title"][:40])

> **확인 포인트** — 기간은 `(20261001, 20261031)`, 1위는 10/28 「지역·필수의료 수가 2.7조 원 우선 투입…」(`mohw_1492342`)입니다. 위 다섯 개 중 넷이 같은 보도자료의 다른 조각이고, 다섯째가 10/1 「루게릭병 전문요양병원 시범사업」입니다.

In [ ]:
top = search("의료 수가 정책", k=5)
assert month_range(TODAY) == (20261001, 20261031)
assert top[0].payload["doc_id"] == "mohw_1492342"
assert [h.payload["doc_id"] for h in top].count("mohw_1492342") == 4

한 보도자료의 조각이 위를 다 차지하면 다른 보도자료가 밀려납니다. 조각을 **보도자료 단위로 묶어** 가장 높은 점수 순으로 놓습니다.

In [ ]:
def search_docs(q, n=3, **kw):
    best = {}
    for h in search(q, k=20, **kw):
        d = h.payload["doc_id"]
        if d not in best:                         # 점수 순으로 오므로 처음 나온 조각이 그 보도자료의 최고점
            best[d] = {"doc_id": d, "date": h.payload["date"], "title": h.payload["title"],
                       "score": round(h.score, 3), "chunks": []}
        best[d]["chunks"].append(h.payload["text"])
    return list(best.values())[:n]

pd.DataFrame(search_docs("노인 돌봄 정책"))[["doc_id", "date", "score", "title"]]

> **확인 포인트** — 「노인 돌봄 정책」의 1위는 10/19 「보험업권과 복지부 산하 노인지원기관이 손잡고 65만명의 취약어르신에게…」(`mohw_1492261`), 2위 10/13 유휴 어린이집의 「주거-돌봄 연계」 공간(`mohw_1492213`), 3위 10/2 「제30회 노인의 날 기념식」입니다.
> 점수 차가 작은 아래 순위는 실행 환경에 따라 바뀔 수 있어, 1위와 2위만 확인합니다.

In [ ]:
docs3 = [d["doc_id"] for d in search_docs("노인 돌봄 정책")]
assert docs3[:2] == ["mohw_1492261", "mohw_1492213"]

조건이 제대로 걸리는지 확인합니다. 9월로 바꾸면 이 자료에는 9월 보도자료가 없으니 아무것도 나오지 않아야 합니다.

In [ ]:
sept = search("노인 돌봄 정책", month=dt.date(2026, 9, 15))
print("9월 결과:", len(sept))
assert len(sept) == 0

## 8단계 · 「이번 달 ○○ 정책 요약해줘」에 답하기

질문에서 「이번 달」 · 「요약해줘」 같은 말을 떼어 주제만 남기고, 관련 보도자료 세 건의 조각을 근거로 모읍니다.
OpenAI 키가 있으면 `gpt-4o-mini` 가 근거만 써서 요약하고, 없으면 근거 문단과 출처를 그대로 보여 줍니다(7주차 RAG).

In [ ]:
KEY = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata             # Colab 왼쪽 열쇠 아이콘(보안 비밀)에 OPENAI_API_KEY 를 넣었다면
    KEY = KEY or userdata.get("OPENAI_API_KEY")
except Exception:
    pass
print("요약 모델:", "gpt-4o-mini" if KEY else "없음 — 근거 문단만 보여 줍니다")

In [ ]:
def topic_of(question):
    q = re.sub(r"(이번\s*달|이달|요즘)", "", question)
    q = re.sub(r"(을|를)?\s*(요약해\s*줘|요약해\s*주세요|알려\s*줘|정리해\s*줘)\s*[.?!]*$", "", q)
    return q.strip()

def ask(question):
    topic = ___⑮___   # ⑮ 질문에서 주제만
    found = search_docs(topic, n=3)
    evidence = "\n\n".join(f"[{i + 1}] {d['title']} ({d['date']})\n" + "\n".join(d["chunks"][:2]) for i, d in enumerate(found))
    if KEY:
        need("openai")
        from openai import OpenAI
        msg = [{"role": "system", "content": "너는 정부 보도자료 요약 도우미야. 아래 근거에 있는 내용만 써서 한국어로 5줄 안팎으로 요약하고, "
                                              "문장 끝에 근거 번호 [1] [2] 를 붙여줘. 근거에 없으면 모른다고 해줘."},
               {"role": "user", "content": f"질문: {question}\n\n근거:\n{evidence}"}]
        answer = OpenAI(api_key=KEY).chat.completions.create(model="gpt-4o-mini", messages=msg, temperature=0).choices[0].message.content
    else:
        answer = "(요약 모델 없음) 관련 보도자료와 근거 문단입니다.\n\n" + "\n\n".join(
            f"[{i + 1}] {d['title']} ({d['date']})\n  {d['chunks'][0][:200]}…" for i, d in enumerate(found))
    return {"topic": topic, "answer": answer, "sources": [d["doc_id"] for d in found]}

r = ask("이번 달 의료 수가 정책 요약해줘")
print("주제:", r["topic"])
print(r["answer"])
print("출처:", r["sources"])

> **확인 포인트** — 주제가 `의료 수가 정책` 으로 줄고, 첫 출처는 `mohw_1492342`(10/28 건강보험 수가 구조 혁신)입니다. 키가 있으면 그 근거만으로 쓴 요약이, 없으면 근거 문단 세 개가 나옵니다.

In [ ]:
assert r["topic"] == "의료 수가 정책" and r["sources"][0] == "mohw_1492342"

다른 질문도 해 봅니다.

In [ ]:
r2 = ask("이번 달 국민연금 정책 알려줘")
print(r2["topic"], r2["sources"])
print(r2["answer"][:600])
assert r2["sources"][0] == "mohw_1492125" and "mohw_1492285" in r2["sources"]

> **확인 포인트** — 주제 `국민연금 정책`, 1위는 10/1 「군복무 기간 전체로 국민연금 크레딧 확대 등 국민연금법, 기초연금법 개정안 국회 본회의 통과」(`mohw_1492125`)이고, 10/20 「제7차 국민연금기금운용위원회 개최」(`mohw_1492285`)도 세 건 안에 듭니다.

> **자주 겪는 오류** — 요약 모델은 근거에 없는 내용을 그럴듯하게 덧붙일 수 있습니다. 시스템 지시에 「근거에 있는 내용만」 · 「근거 번호를 붙여」를 넣고, 답 아래에 출처 목록을 함께 보여 주세요. 출처가 없는 요약은 보도자료 검색기로 쓰기 어렵습니다.

## 9단계 · 정리와 더 해 보기

띄운 MinIO 를 끕니다. 끄지 않으면 노트북을 다시 실행할 때 포트가 이미 쓰이고 있다고 나옵니다.

In [ ]:
if minio_proc:
    minio_proc.___⑯___()   # ⑯ 프로세스 끄기
    minio_proc.wait(timeout=10)
    print("MinIO 종료:", minio_proc.returncode is not None)

- 질문 상자 하나와 답 · 출처를 보여 주는 화면을 Streamlit(8주차)으로 만들어 보세요. `ask()` 를 그대로 부르면 됩니다.
- 매일 아침 2단계 수집 → 3~6단계를 이어 돌리는 Airflow DAG(8주차)를 짜 보세요. 이미 넣은 보도자료(`doc_id`)는 건너뛰게 하는 것이 핵심입니다.
- 7주차 7-5 하이브리드 검색처럼 「국민연금」 같은 고유명사를 낱말로도 찾게 하면 순위가 어떻게 바뀌나요? 4단계 주의의 띄어진 글자를 먼저 정제해 보세요.
- 다른 부처(과학기술정보통신부 · 행정안전부 등) 보도자료도 같은 버킷에 `msit/2026-10/` 처럼 넣고, 「부처」 payload 로 거르게 해 보세요. M4 에서 모은 RSS 목록이 출발점이 됩니다.

## 정답

① `findall` — 규칙에 맞는 조각을 모두 찾기
② `Popen` — 끝나기를 기다리지 않고 띄우기
③ `put_object` — 객체 하나 올리기
④ `list_objects_v2` — 객체 목록 API
⑤ `make_bucket` — 버킷 만들기
⑥ `extract_text` — 쪽에서 글자 뽑기
⑦ `get("press-raw", key)` — 버킷에서 원본 읽기
⑧ `500` — 조각 최대 글자 수
⑨ `50` — 겹치는 글자 수
⑩ `encode` — 글자를 벡터로
⑪ `768` — 벡터 길이
⑫ `COSINE` — 방향이 얼마나 같은지
⑬ `Range` — 이상 · 이하 조건
⑭ `flt` — 날짜 조건
⑮ `topic_of(question)` — 질문에서 주제만
⑯ `terminate` — 프로세스 끄기